# 05 · Comparación de modelos y selección del ganador
Se entrenan los tres modelos con la **misma partición** y se comparan con las mismas métricas.
Los modelos que todavía no estén implementados se saltan automáticamente.

In [ ]:
# Configuración: permite importar el código común de src/ desde la carpeta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
plt.style.use("ggplot")
sns.set_palette("muted")

from src import config, data, evaluation
from src.models import MODELOS
from src.training import entrenar_y_evaluar, exportar_final

In [ ]:
df = data.cargar_dataset()

pipelines, resultados = {}, {}
for clave, mod in MODELOS.items():
    try:
        pipe, m, (X_test, y_test) = entrenar_y_evaluar(clave, df)
    except NotImplementedError as ex:
        print(f"⏭  {mod.NOMBRE}: {ex}")
        continue
    pipelines[mod.NOMBRE] = pipe
    resultados[mod.NOMBRE] = m

## 1. Tabla comparativa (clase phishing)

In [ ]:
tabla = pd.DataFrame(resultados).T[["precision", "recall", "f1", "auc", "accuracy"]].astype(float).round(4)
display(tabla.style.highlight_max(axis=0, color="#cfe3ff"))

## 2. Curvas ROC superpuestas

In [ ]:
evaluation.graficar_roc(pipelines, X_test, y_test, "Curvas ROC — comparación")
plt.show()

## 3. Matrices de confusión

In [ ]:
for nombre, m in resultados.items():
    evaluation.graficar_matriz_confusion(m, nombre)
    plt.show()

## 4. Selección del modelo ganador
TODO(equipo): justificar. En ciberseguridad un **Recall** alto en la clase 1 minimiza las víctimas,
y una **Precision** adecuada evita bloquear sitios legítimos.

Después de decidir, cambiar `MODELO_POR_DEFECTO` en `src/config.py` por la clave del ganador
(`"lr"`, `"rf"` o `"svm"`) para que la app lo use por defecto.

In [ ]:
GANADOR = None   # TODO(equipo): "lr", "rf" o "svm"

if GANADOR:
    print("Exportado:", exportar_final(GANADOR, df))